# Spark on Kubernetes - ETL Pipeline Demo

Run one cell at a time and observe real-time changes on the Dashboard.

Execute each cell and watch what happens on the Dashboard!

## Step 1: Create SparkSession with Dynamic Allocation

This step transforms the Jupyter Pod into a Spark Driver with dynamic executor allocation.

**Dashboard Expectation:** Driver is running (this Jupyter Pod), Spark API connection is successful, no executors yet (will be spawned on demand)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import socket

driver_host = socket.gethostbyname(socket.gethostname())

spark = (
    SparkSession.builder
    .appName("Interactive-ETL-Demo")
    .config("spark.master", "k8s://https://kubernetes.default.svc")
    .config("spark.submit.deployMode", "client")
    .config("spark.kubernetes.container.image", "spark-etl:latest")
    .config("spark.kubernetes.container.image.pullPolicy", "Never")
    .config("spark.kubernetes.namespace", "spark-demo")
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.dynamicAllocation.initialExecutors", "1")
    .config("spark.dynamicAllocation.minExecutors", "0")
    .config("spark.dynamicAllocation.maxExecutors", "4")
    .config("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.driver.host", driver_host)
    .config("spark.driver.port", "29413")
    .config("spark.driver.blockManager.port", "29414")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.port.maxRetries", "0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-svc.spark-demo.svc.cluster.local:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession created successfully! App ID: {spark.sparkContext.applicationId}")
print(f"Mode: Client (Jupyter = Driver)")
print(f"Driver Host: {driver_host}")
print(f"Spark UI: Check the Dashboard for real-time monitoring")

## Step 2: Extract - Read Data from MinIO

This is an Action (count) that triggers real execution.

**Dashboard Expectation:**
- Initial executor pod will be created (K8s spawns pod based on dynamic allocation)
- Data flows from MinIO toward executor (reading data)
- Watch the executor count on the Dashboard

In [ ]:
# Read CSV (this is a Transformation, not executed yet)
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("s3a://spark-data/raw/")
)

print("Schema:")
df_raw.printSchema()

# count() is an Action - triggers real execution! Check Dashboard!
count = df_raw.count()
print(f"\nLoaded {count} raw data records")

## Step 3: Transform - Data Cleaning (Narrow Dependency)

Filter and withColumn operations are Narrow Dependencies.

**Dashboard Expectation:** Executors are processing data without Shuffle - each record is processed independently on its partition

In [ ]:
# These are all Transformations, defined but not executed yet
df_clean = (
    df_raw
    .filter(F.col("product").isNotNull())
    .filter(F.col("category").isNotNull())
    .filter(F.col("region").isNotNull())
    .filter(F.col("quantity") > 0)
    .withColumn("total_amount", F.col("quantity") * F.col("unit_price"))
)

# count() is an Action - triggers execution! Check Dashboard!
clean_count = df_clean.count()
removed = count - clean_count
print(f"Cleaning complete: Removed {removed} dirty records, {clean_count} records remaining")

## Step 4: Transform - GroupBy Aggregation (Shuffle!)

GroupBy is a Wide Dependency that triggers Shuffle.

**Dashboard Expectation:**
- Data exchanges between executors (records with same key move together)
- Event Log shows Shuffle events
- You may see executor count increase to handle shuffle traffic

In [ ]:
# groupBy triggers Shuffle!
# Records with same region must be colocated on same executor
df_region = (
    df_clean
    .groupBy("region", "category")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value"),
    )
    .orderBy(F.desc("total_revenue"))
)

# show() is an Action - triggers Shuffle! Check Dashboard!
print("Region x Category Sales Statistics:")
df_region.show(20, truncate=False)

## Step 5: Load - Write Results to MinIO (Parquet)

**Dashboard Expectation:**
- Data flows from executors back to MinIO (writing results)
- Write operations complete and task finishes

In [ ]:
# write is an Action - triggers execution!
df_clean.write.mode("overwrite").parquet("s3a://spark-data/processed/clean_sales/")
print("Cleaned data written successfully")

df_region.write.mode("overwrite").parquet("s3a://spark-data/processed/region_stats/")
print("Region statistics written successfully")

## Step 6: Verify Results

Read back the Parquet files to confirm data was written correctly.

In [ ]:
# Read Parquet back to verify
verify = spark.read.parquet("s3a://spark-data/processed/region_stats/")
print(f"Verification: region_stats contains {verify.count()} records")
verify.show()

## Step 7: Close SparkSession

**Dashboard Expectation:** All executor pods will be terminated, Spark API connection closes, driver is shut down

In [ ]:
spark.stop()
print("SparkSession closed")
print("   Check Dashboard - all executor pods should be gone")